# LOTTO 5/35 - DASHBOARD PHÂN TÍCH DỰ BÁO
## Phân tích Số liệu, Thống kê, Dự báo & Biểu đồ

**Mục đích:**
- Phân tích phân bố số từ dữ liệu xổ số Việt Nam (Lotto 5/35)
- Dự báo những số có khả năng xuất hiện cao
- Hiển thị biểu đồ thống kê chi tiết
- So sánh với dữ liệu lịch sử

**Lưu ý:** Phân tích này dựa trên xác suất thống kê, không đảm bảo 100% chính xác

## 1. CÀI ĐẶT THƯ VIỆN

In [ ]:
# Cài đặt các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Cấu hình hiển thị
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Hỗ trợ tiếng Việt
plt.rcParams['font.family'] = 'DejaVu Sans'

print("✓ Các thư viện đã được tải thành công!")

## 2. TẢI VÀ TIỀN XỬ LÝ DỮ LIỆU

In [ ]:
# Đọc file CSV từ URL hoặc local
url = 'https://raw.githubusercontent.com/PhDLeToanThang/ai-ml-dl/refs/heads/main/aipl/lotto535.csv'

try:
    # Thử đọc từ URL
    df = pd.read_csv(url, encoding='utf-8')
    print("✓ Dữ liệu được tải từ GitHub URL")
except:
    # Nếu lỗi, đọc từ file local
    df = pd.read_csv('lotto535.csv', encoding='utf-8')
    print("✓ Dữ liệu được tải từ file local")

# Hiển thị thông tin cơ bản
print(f"\nDữ liệu có {len(df)} bản ghi, {len(df.columns)} cột")
print(f"\nCác cột: {df.columns.tolist()}")
print(f"\nDữ liệu từ: {df['Ngày'].iloc[0]} đến {df['Ngày'].iloc[-1]}")
print(f"\nDữ liệu mẫu:")
print(df.head(10))

In [ ]:
# Tiền xử lý dữ liệu
df['Ngày'] = pd.to_datetime(df['Ngày'], format='%d/%m/%Y')
df = df.sort_values('Ngày').reset_index(drop=True)

# Kiểm tra dữ liệu
print("Kiểm tra dữ liệu:")
print(df.info())
print(f"\nGiá trị missing: {df.isnull().sum().sum()}")
print(f"\nThống kê cơ bản:")
print(df.describe())

## 3. PHÂN TÍCH TẦN SUẤT SỐ

In [ ]:
# Trích xuất tất cả các số từ các cột s1 đến s5
number_columns = ['s1', 's2', 's3', 's4', 's5']
all_numbers = []

for col in number_columns:
    all_numbers.extend(df[col].values.tolist())

# Tính tần suất
number_frequency = Counter(all_numbers)
frequency_df = pd.DataFrame([
    {'Số': num, 'Tần suất': freq, 'Phần trăm': f"{(freq / len(all_numbers) * 100):.2f}%"}
    for num, freq in sorted(number_frequency.items())
]).sort_values('Tần suất', ascending=False)

print(f"\nTổng số lần quay: {len(all_numbers)}")
print(f"Số lần xuất hiện trung bình mỗi số: {len(all_numbers) / 35:.2f}")
print(f"\nTop 10 số xuất hiện nhiều nhất:")
print(frequency_df.head(10).to_string(index=False))
print(f"\nTop 10 số xuất hiện ít nhất:")
print(frequency_df.tail(10).to_string(index=False))

## 4. PHÂN TÍCH DỰ BÁO THEO CÁC THUẬT TOÁN

In [ ]:
# ===== THUẬT TOÁN 1: FREQUENCY ANALYSIS (Phân tích tần suất) =====
class FrequencyAnalysis:
    def __init__(self, frequency_dict):
        self.freq = frequency_dict
        self.total = sum(frequency_dict.values())
    
    def get_hot_numbers(self, top_n=10):
        """Các số 'nóng' - xuất hiện nhiều"""
        return sorted(self.freq.items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    def get_cold_numbers(self, top_n=10):
        """Các số 'lạnh' - xuất hiện ít"""
        return sorted(self.freq.items(), key=lambda x: x[1])[:top_n]
    
    def get_probability(self, number):
        """Xác suất xuất hiện của một số"""
        return self.freq.get(number, 0) / self.total

fa = FrequencyAnalysis(number_frequency)

print("="*60)
print("THUẬT TOÁN 1: PHÂN TÍCH TẦN SUẤT")
print("="*60)
hot_nums = fa.get_hot_numbers(10)
print("\nTop 10 SỐ NÓNG (Hot Numbers - Xuất hiện nhiều):")
for num, freq in hot_nums:
    prob = fa.get_probability(num)
    print(f"  Số {num:2d}: {freq:3d} lần ({prob*100:5.2f}%)")

cold_nums = fa.get_cold_numbers(10)
print("\nTop 10 SỐ LẠNH (Cold Numbers - Xuất hiện ít):")
for num, freq in cold_nums:
    prob = fa.get_probability(num)
    print(f"  Số {num:2d}: {freq:3d} lần ({prob*100:5.2f}%)")

In [ ]:
# ===== THUẬT TOÁN 2: OVERDUE ANALYSIS (Phân tích số quá hạn) =====
class OverdueAnalysis:
    def __init__(self, df, number_columns):
        self.df = df
        self.number_columns = number_columns
    
    def get_last_appearance(self):
        """Lần xuất hiện cuối cùng của mỗi số"""
        last_app = {}
        for idx, row in self.df.iterrows():
            for col in self.number_columns:
                num = row[col]
                last_app[num] = idx
        return last_app
    
    def get_overdue_numbers(self):
        """Các số chưa xuất hiện lâu - có khả năng xuất hiện cao"""
        last_app = self.get_last_appearance()
        current_draw = len(self.df) - 1
        
        overdue = {}
        for num in range(1, 36):
            if num in last_app:
                draws_ago = current_draw - last_app[num]
                overdue[num] = draws_ago
            else:
                overdue[num] = current_draw + 1
        
        return sorted(overdue.items(), key=lambda x: x[1], reverse=True)

oa = OverdueAnalysis(df, number_columns)
overdue_nums = oa.get_overdue_numbers()

print("="*60)
print("THUẬT TOÁN 2: PHÂN TÍCH SỐ QUOHẠN (Overdue Numbers)")
print("="*60)
print("\nTop 10 SỐ QUÁHẠN (Chưa xuất hiện lâu, cần xuất hiện):")
for num, draws_ago in overdue_nums[:10]:
    print(f"  Số {num:2d}: {draws_ago:3d} kỳ quay không xuất hiện")

In [ ]:
# ===== THUẬT TOÁN 3: PAIR ANALYSIS (Phân tích cặp số) =====
class PairAnalysis:
    def __init__(self, df, number_columns):
        self.df = df
        self.number_columns = number_columns
    
    def get_pair_frequency(self):
        """Tính tần suất của các cặp số xuất hiện trong cùng kỳ"""
        pairs = defaultdict(int)
        
        for idx, row in self.df.iterrows():
            numbers = sorted([row[col] for col in self.number_columns])
            # Tạo các cặp từ 5 số
            for i in range(len(numbers)):
                for j in range(i+1, len(numbers)):
                    pair = tuple(sorted([numbers[i], numbers[j]]))
                    pairs[pair] += 1
        
        return pairs
    
    def get_number_pair_frequency(self, number):
        """Các số hay xuất hiện cùng với một số nhất định"""
        pairs = self.get_pair_frequency()
        companions = defaultdict(int)
        
        for (n1, n2), freq in pairs.items():
            if n1 == number:
                companions[n2] += freq
            elif n2 == number:
                companions[n1] += freq
        
        return sorted(companions.items(), key=lambda x: x[1], reverse=True)

pa = PairAnalysis(df, number_columns)
pair_freq = pa.get_pair_frequency()

print("="*60)
print("THUẬT TOÁN 3: PHÂN TÍCH CẶP SỐ (Pair Analysis)")
print("="*60)
print(f"\nTổng số cặp số khác nhau: {len(pair_freq)}")
print("\nTop 10 cặp số xuất hiện nhiều nhất cùng nhau:")

top_pairs = sorted(pair_freq.items(), key=lambda x: x[1], reverse=True)[:10]
for (n1, n2), freq in top_pairs:
    print(f"  ({n1:2d}, {n2:2d}): {freq:3d} lần")

In [ ]:
# ===== THUẬT TOÁN 4: STATISTICAL PREDICTION (Dự báo thống kê) =====
class StatisticalPrediction:
    def __init__(self, frequency_dict, total_draws):
        self.freq = frequency_dict
        self.total_draws = total_draws
        self.expected_frequency = total_draws * 5 / 35  # Mỗi số lý thuyết xuất hiện bao nhiêu lần
    
    def chi_square_deviation(self, number):
        """Độ lệch chi bình phương"""
        observed = self.freq.get(number, 0)
        expected = self.expected_frequency
        if expected > 0:
            return ((observed - expected) ** 2) / expected
        return 0
    
    def get_deviation_score(self, number):
        """Điểm độ lệch (âm = ít xuất hiện, dương = nhiều xuất hiện)"""
        observed = self.freq.get(number, 0)
        expected = self.expected_frequency
        return observed - expected
    
    def get_prediction_score(self, number):
        """Điểm dự báo (0-100) - số nào cao hơn thì càng có khả năng xuất hiện"""
        deviation = self.get_deviation_score(number)
        # Normalize thành 0-100
        all_deviations = [self.get_deviation_score(n) for n in range(1, 36)]
        min_dev = min(all_deviations)
        max_dev = max(all_deviations)
        
        if max_dev == min_dev:
            return 50
        return ((deviation - min_dev) / (max_dev - min_dev)) * 100

sp = StatisticalPrediction(number_frequency, len(df))

print("="*60)
print("THUẬT TOÁN 4: DỰ BÁO THỐNG KÊ (Statistical Prediction)")
print("="*60)
print(f"\nTổng số kỳ quay: {len(df)}")
print(f"Tần suất xuất hiện kỳ vọng mỗi số: {sp.expected_frequency:.2f} lần")
print(f"\nTop 10 số có điểm dự báo cao nhất:")

predictions = [(num, sp.get_prediction_score(num), sp.get_deviation_score(num)) 
               for num in range(1, 36)]
predictions.sort(key=lambda x: x[1], reverse=True)

for num, score, deviation in predictions[:10]:
    freq = number_frequency.get(num, 0)
    print(f"  Số {num:2d}: Điểm dự báo {score:6.2f}/100 | Tần suất: {freq:2d} | Độ lệch: {deviation:+6.2f}")

In [ ]:
# ===== THUẬT TOÁN 5: POSITIONAL PATTERN ANALYSIS (Phân tích mẫu vị trí) =====
class PositionalAnalysis:
    def __init__(self, df, position_columns):
        self.df = df
        self.position_columns = position_columns
    
    def get_position_frequency(self):
        """Tần suất của các số ở từng vị trí"""
        position_freq = {}
        for pos_name in self.position_columns:
            numbers = self.df[pos_name].values
            position_freq[pos_name] = Counter(numbers)
        return position_freq
    
    def get_hot_numbers_by_position(self, position, top_n=5):
        """Các số nóng ở vị trí nhất định"""
        pos_freq = self.get_position_frequency()
        if position in pos_freq:
            return sorted(pos_freq[position].items(), key=lambda x: x[1], reverse=True)[:top_n]
        return []

pa_pos = PositionalAnalysis(df, number_columns)

print("="*60)
print("THUẬT TOÁN 5: PHÂN TÍCH MẪU VỊ TRÍ (Positional Pattern)")
print("="*60)

for position in number_columns:
    hot_at_pos = pa_pos.get_hot_numbers_by_position(position, 5)
    print(f"\nVị trí {position} - Top 5 số nóng:")
    for num, freq in hot_at_pos:
        print(f"  Số {num:2d}: {freq} lần")

In [ ]:
# ===== THUẬT TOÁN 6: SUM & AVERAGE ANALYSIS (Phân tích tổng & trung bình) =====
class SumAverageAnalysis:
    def __init__(self, df, number_columns):
        self.df = df.copy()
        self.number_columns = number_columns
        self.df['sum'] = self.df[number_columns].sum(axis=1)
        self.df['average'] = self.df[number_columns].mean(axis=1)
    
    def get_stats(self):
        """Thống kê về tổng và trung bình"""
        return {
            'sum_min': self.df['sum'].min(),
            'sum_max': self.df['sum'].max(),
            'sum_mean': self.df['sum'].mean(),
            'sum_std': self.df['sum'].std(),
            'avg_min': self.df['average'].min(),
            'avg_max': self.df['average'].max(),
            'avg_mean': self.df['average'].mean(),
            'avg_std': self.df['average'].std(),
        }
    
    def predict_sum_range(self):
        """Dự báo khoảng tổng số"""
        stats = self.get_stats()
        mean = stats['sum_mean']
        std = stats['sum_std']
        # Khoảng 1 sigma (68% xác suất)
        return {
            'likely_range': (mean - std, mean + std),
            'confident_range': (mean - 2*std, mean + 2*std),
        }

sa = SumAverageAnalysis(df, number_columns)
sa_stats = sa.get_stats()
sa_predict = sa.predict_sum_range()

print("="*60)
print("THUẬT TOÁN 6: PHÂN TÍCH TỔNG & TRUNG BÌNH (Sum & Average)")
print("="*60)
print(f"\nTHỐNG KÊ TỔNG SỐ:")
print(f"  Min: {sa_stats['sum_min']:.0f}")
print(f"  Max: {sa_stats['sum_max']:.0f}")
print(f"  Trung bình: {sa_stats['sum_mean']:.2f}")
print(f"  Độ lệch chuẩn: {sa_stats['sum_std']:.2f}")
print(f"\nTHỐNG KÊ TRUNG BÌNH:")
print(f"  Min: {sa_stats['avg_min']:.2f}")
print(f"  Max: {sa_stats['avg_max']:.2f}")
print(f"  Trung bình: {sa_stats['avg_mean']:.2f}")
print(f"  Độ lệch chuẩn: {sa_stats['avg_std']:.2f}")
print(f"\nDỰ BÁO KHOẢNG TỔNG SỐ:")
print(f"  Khoảng khả dĩ (68% xác suất): {sa_predict['likely_range'][0]:.0f} - {sa_predict['likely_range'][1]:.0f}")
print(f"  Khoảng tự tin (95% xác suất): {sa_predict['confident_range'][0]:.0f} - {sa_predict['confident_range'][1]:.0f}")

In [ ]:
# ===== THUẬT TOÁN 7: GAP ANALYSIS (Phân tích khoảng cách số) =====
class GapAnalysis:
    def __init__(self, df, number_columns):
        self.df = df
        self.number_columns = number_columns
    
    def get_gap_frequency(self):
        """Tính tần suất các khoảng cách giữa các số"""
        gaps = defaultdict(int)
        
        for idx, row in self.df.iterrows():
            numbers = sorted([row[col] for col in self.number_columns])
            # Tính khoảng cách giữa các số liên tiếp
            for i in range(len(numbers) - 1):
                gap = numbers[i+1] - numbers[i]
                gaps[gap] += 1
        
        return gaps
    
    def get_gap_stats(self):
        """Thống kê về khoảng cách"""
        gaps = self.get_gap_frequency()
        gap_values = list(gaps.values())
        
        return {
            'total_gaps': len(gaps),
            'gaps': gaps,
            'avg_gap': np.mean(gap_values),
            'most_common_gap': max(gaps, key=gaps.get),
        }

ga = GapAnalysis(df, number_columns)
ga_stats = ga.get_gap_stats()

print("="*60)
print("THUẬT TOÁN 7: PHÂN TÍCH KHOẢNG CÁCH SỐ (Gap Analysis)")
print("="*60)
print(f"\nTổng các khoảng cách khác nhau: {ga_stats['total_gaps']}")
print(f"Khoảng cách trung bình: {ga_stats['avg_gap']:.2f}")
print(f"Khoảng cách phổ biến nhất: {ga_stats['most_common_gap']}")
print(f"\nTop 10 khoảng cách xuất hiện nhiều nhất:")

gaps_sorted = sorted(ga_stats['gaps'].items(), key=lambda x: x[1], reverse=True)[:10]
for gap, freq in gaps_sorted:
    print(f"  Khoảng cách {gap:2d}: {freq:3d} lần")

## 5. TỔNG HỢP DỰ BÁO TOP NUMBERS

In [ ]:
# Tổng hợp điểm từ tất cả các thuật toán
class CombinedPrediction:
    def __init__(self, freq_analysis, overdue_analysis, stat_prediction, 
                 pair_analysis, positional_analysis, total_numbers=35):
        self.fa = freq_analysis
        self.oa = overdue_analysis
        self.sp = stat_prediction
        self.pa = pair_analysis
        self.pa_pos = positional_analysis
        self.total_numbers = total_numbers
    
    def calculate_combined_score(self):
        """Tính điểm tổng hợp cho mỗi số"""
        scores = {}
        
        # Lấy điểm từ các thuật toán khác nhau
        hot_nums = dict(self.fa.get_hot_numbers(35))
        overdue_nums = dict(self.oa.get_overdue_numbers())
        
        for num in range(1, self.total_numbers + 1):
            score = 0
            
            # 1. Điểm tần suất (25%)
            freq_score = (hot_nums.get(num, 0) / max(hot_nums.values(), default=1)) * 25 if hot_nums else 0
            
            # 2. Điểm quá hạn (25%)
            overdue_dict = dict(overdue_nums)
            overdue_score = (overdue_dict.get(num, 0) / max(overdue_dict.values(), default=1)) * 25 if overdue_dict else 0
            
            # 3. Điểm dự báo thống kê (30%)
            stat_score = self.sp.get_prediction_score(num) * 0.3
            
            # 4. Điểm từ cặp số (10%)
            companions = self.pa.get_number_pair_frequency(num)
            companion_score = (len(companions) / self.total_numbers) * 10 if companions else 0
            
            # 5. Điểm vị trí (10%)
            position_score = 10  # Mặc định
            
            score = freq_score + overdue_score + stat_score + companion_score + position_score
            scores[num] = score
        
        return scores
    
    def get_top_predictions(self, top_n=10):
        """Lấy top N số dự báo tốt nhất"""
        scores = self.calculate_combined_score()
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

cp = CombinedPrediction(fa, oa, sp, pa, pa_pos, total_numbers=35)
top_predictions = cp.get_top_predictions(10)

print("="*60)
print("🎯 TỔNG HỢP DỰ BÁO TOP 10 SỐ CÓ KHẢ NĂNG CAO")
print("="*60)
print("\nTop 10 số dự báo tốt nhất (dựa trên tổng hợp các thuật toán):")
print(f"{'Rank':<6} {'Số':<6} {'Điểm Dự Báo':<15} {'Tần Suất':<12} {'Quá Hạn':<10}")
print("-" * 60)

for rank, (num, score) in enumerate(top_predictions, 1):
    freq = number_frequency.get(num, 0)
    overdue_dict = dict(oa.get_overdue_numbers())
    overdue = overdue_dict.get(num, 0)
    print(f"{rank:<6} {num:<6} {score:<15.2f} {freq:<12} {overdue:<10}")

In [ ]:
# In ra tất cả số với điểm dự báo
all_scores = cp.calculate_combined_score()
all_predictions = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)

print("\nBẢNG ĐIỂM DỰ BÁO ĐẦY ĐỦ CẢ 35 SỐ:")
print(f"{'Số':<6} {'Điểm':<10} {'Tần Suất':<12} {'Quá Hạn':<10} {'Status':<15}")
print("-" * 65)

for num, score in all_predictions:
    freq = number_frequency.get(num, 0)
    overdue_dict = dict(oa.get_overdue_numbers())
    overdue = overdue_dict.get(num, 0)
    
    if score > 70:
        status = "🔥 Rất Hot"
    elif score > 60:
        status = "🔥 Hot"
    elif score > 50:
        status = "⚠️ Trung Bình"
    else:
        status = "❄️ Lạnh"
    
    print(f"{num:<6} {score:<10.2f} {freq:<12} {overdue:<10} {status:<15}")

## 6. BIỂU ĐỒ VỀ HIỂN THỊTRỰC QUAN

In [ ]:
# BIỂU ĐỒ 1: Tần suất xuất hiện của tất cả các số
fig, ax = plt.subplots(figsize=(18, 6))

nums = sorted(frequency_df['Số'].unique())
freqs = [number_frequency.get(n, 0) for n in nums]

colors = ['#ff6b6b' if f > frequency_df['Tần suất'].mean() else '#4ecdc4' for f in freqs]

bars = ax.bar(nums, freqs, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(y=frequency_df['Tần suất'].mean(), color='red', linestyle='--', label='Trung bình', linewidth=2)

ax.set_xlabel('Số', fontsize=12, fontweight='bold')
ax.set_ylabel('Tần Suất', fontsize=12, fontweight='bold')
ax.set_title('BIỂU ĐỒ TẦN SUẤT XUẤT HIỆN CỦA TẤT CẢ CÁC SỐ (1-35)', fontsize=14, fontweight='bold')
ax.set_xticks(nums)
ax.grid(axis='y', alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print("✓ Biểu đồ 1 hoàn thành")

In [ ]:
# BIỂU ĐỒ 2: Top 15 số nóng vs số lạnh
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Top hot numbers
hot_data = frequency_df.head(15)
ax1.barh(hot_data['Số'].astype(str), hot_data['Tần suất'], color='#ff6b6b', edgecolor='black')
ax1.set_xlabel('Tần Suất', fontsize=11, fontweight='bold')
ax1.set_title('Top 15 SỐ NÓNG (Hot Numbers)', fontsize=12, fontweight='bold')
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)

# Top cold numbers
cold_data = frequency_df.tail(15)
ax2.barh(cold_data['Số'].astype(str), cold_data['Tần suất'], color='#4ecdc4', edgecolor='black')
ax2.set_xlabel('Tần Suất', fontsize=11, fontweight='bold')
ax2.set_title('Top 15 SỐ LẠNH (Cold Numbers)', fontsize=12, fontweight='bold')
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Biểu đồ 2 hoàn thành")

In [ ]:
# BIỂU ĐỒ 3: Điểm dự báo tổng hợp
fig, ax = plt.subplots(figsize=(18, 6))

all_nums = list(range(1, 36))
all_scores_list = [all_scores[n] for n in all_nums]

colors_pred = ['#ff6b6b' if s > 70 else '#ffa400' if s > 60 else '#4ecdc4' if s > 50 else '#95a5a6' 
               for s in all_scores_list]

bars = ax.bar(all_nums, all_scores_list, color=colors_pred, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Số', fontsize=12, fontweight='bold')
ax.set_ylabel('Điểm Dự Báo', fontsize=12, fontweight='bold')
ax.set_title('BIỂU ĐỒ ĐIỂM DỰ BÁO TỔNG HỢP CỦA TẤT CẢ CÁC SỐ', fontsize=14, fontweight='bold')
ax.set_xticks(all_nums)
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

# Thêm legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#ff6b6b', edgecolor='black', label='Rất Hot (>70)'),
    Patch(facecolor='#ffa400', edgecolor='black', label='Hot (60-70)'),
    Patch(facecolor='#4ecdc4', edgecolor='black', label='Trung Bình (50-60)'),
    Patch(facecolor='#95a5a6', edgecolor='black', label='Lạnh (<50)')
]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

print("✓ Biểu đồ 3 hoàn thành")

In [ ]:
# BIỂU ĐỒ 4: Tổng số qua các kỳ quay
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(range(len(sa.df)), sa.df['sum'], marker='o', linestyle='-', linewidth=1, markersize=3, label='Tổng số')
ax.axhline(y=sa_stats['sum_mean'], color='red', linestyle='--', label=f"Trung bình: {sa_stats['sum_mean']:.1f}", linewidth=2)
ax.fill_between(range(len(sa.df)), 
                sa_stats['sum_mean'] - sa_stats['sum_std'], 
                sa_stats['sum_mean'] + sa_stats['sum_std'], 
                alpha=0.2, label='1 Sigma')

ax.set_xlabel('Kỳ Quay', fontsize=12, fontweight='bold')
ax.set_ylabel('Tổng Số', fontsize=12, fontweight='bold')
ax.set_title('BIỂU ĐỒ TỔNG SỐ QUA CÁC KỲ QUAY', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print("✓ Biểu đồ 4 hoàn thành")

In [ ]:
# BIỂU ĐỒ 5: Phân bố tần suất (Histogram)
fig, ax = plt.subplots(figsize=(14, 6))

ax.hist(freqs, bins=15, color='#3498db', edgecolor='black', alpha=0.7)
ax.axvline(x=frequency_df['Tần suất'].mean(), color='red', linestyle='--', 
           label=f"Trung bình: {frequency_df['Tần suất'].mean():.1f}", linewidth=2)
ax.axvline(x=frequency_df['Tần suất'].median(), color='green', linestyle='--', 
           label=f"Trung vị: {frequency_df['Tần suất'].median():.1f}", linewidth=2)

ax.set_xlabel('Tần Suất', fontsize=12, fontweight='bold')
ax.set_ylabel('Số Lượng', fontsize=12, fontweight='bold')
ax.set_title('PHÂN BỐ TẦN SUẤT CỦA CÁC SỐ', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print("✓ Biểu đồ 5 hoàn thành")

In [ ]:
# BIỂU ĐỒ 6: Heatmap tần suất vị trí
fig, ax = plt.subplots(figsize=(12, 6))

# Tạo ma trận vị trí x số
position_matrix = np.zeros((len(number_columns), 35))

for pos_idx, pos_col in enumerate(number_columns):
    for num in range(1, 36):
        position_matrix[pos_idx, num-1] = number_frequency.get(num, 0)

sns.heatmap(position_matrix, cmap='YlOrRd', annot=False, fmt='d', 
            xticklabels=range(1, 36), yticklabels=number_columns, ax=ax, cbar_kws={'label': 'Tần Suất'})
ax.set_title('HEATMAP TẦN SUẤT CỦA CÁC SỐ', fontsize=14, fontweight='bold')
ax.set_xlabel('Số', fontsize=12, fontweight='bold')
ax.set_ylabel('Vị Trí', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Biểu đồ 6 hoàn thành")

In [ ]:
# BIỂU ĐỒ 7: Top 20 cặp số xuất hiện cùng nhau
fig, ax = plt.subplots(figsize=(14, 8))

top_pairs_20 = sorted(pair_freq.items(), key=lambda x: x[1], reverse=True)[:20]
pair_labels = [f"({p[0]}, {p[1]})" for p, _ in top_pairs_20]
pair_counts = [count for _, count in top_pairs_20]

ax.barh(pair_labels, pair_counts, color='#e74c3c', edgecolor='black')
ax.set_xlabel('Tần Suất', fontsize=12, fontweight='bold')
ax.set_title('TOP 20 CẶP SỐ XUẤT HIỆN CÙNG NHAU', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Biểu đồ 7 hoàn thành")

In [ ]:
# BIỂU ĐỒ 8: Phân bố khoảng cách giữa các số
fig, ax = plt.subplots(figsize=(14, 6))

gap_data = sorted(ga_stats['gaps'].items())
gap_labels = [str(g) for g, _ in gap_data]
gap_counts = [count for _, count in gap_data]

ax.bar(gap_labels, gap_counts, color='#9b59b6', edgecolor='black', alpha=0.7)
ax.set_xlabel('Khoảng Cách', fontsize=12, fontweight='bold')
ax.set_ylabel('Tần Suất', fontsize=12, fontweight='bold')
ax.set_title('PHÂN BỐ KHOẢNG CÁCH GIỮA CÁC SỐ LIÊN TIẾP', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Biểu đồ 8 hoàn thành")

In [ ]:
# BIỂU ĐỒ 9: So sánh tần suất vs dự báo
fig, ax = plt.subplots(figsize=(16, 6))

all_nums = list(range(1, 36))
freqs_list = [number_frequency.get(n, 0) for n in all_nums]
predictions_list = [all_scores[n] for n in all_nums]

# Normalize predictions để so sánh
predictions_normalized = [p * (max(freqs_list) / max(predictions_list)) for p in predictions_list]

x = np.arange(len(all_nums))
width = 0.35

ax.bar(x - width/2, freqs_list, width, label='Tần Suất Thực', color='#3498db', edgecolor='black')
ax.bar(x + width/2, predictions_normalized, width, label='Dự Báo', color='#e74c3c', edgecolor='black')

ax.set_xlabel('Số', fontsize=12, fontweight='bold')
ax.set_ylabel('Tần Suất / Dự Báo', fontsize=12, fontweight='bold')
ax.set_title('SO SÁNH TẦN SUẤT THỰC TẾ VS DỰ BÁO', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(all_nums)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Biểu đồ 9 hoàn thành")

In [ ]:
# BIỂU ĐỒ 10: Pie chart - Phân loại số nóng, lạnh
fig, ax = plt.subplots(figsize=(10, 8))

very_hot = len([s for s in all_scores_list if s > 70])
hot = len([s for s in all_scores_list if 60 < s <= 70])
medium = len([s for s in all_scores_list if 50 < s <= 60])
cold = len([s for s in all_scores_list if s <= 50])

sizes = [very_hot, hot, medium, cold]
labels = [f'Rất Hot (>70)\n{very_hot} số', 
          f'Hot (60-70)\n{hot} số', 
          f'Trung Bình (50-60)\n{medium} số', 
          f'Lạnh (<50)\n{cold} số']
colors_pie = ['#ff6b6b', '#ffa400', '#4ecdc4', '#95a5a6']
explode = (0.05, 0.05, 0.05, 0.05)

ax.pie(sizes, explode=explode, labels=labels, colors=colors_pie, autopct='%1.1f%%',
       shadow=True, startangle=90, textprops={'fontsize': 11, 'weight': 'bold'})
ax.set_title('PHÂN LOẠI SỐ THEO ĐIỂM DỰ BÁO', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Biểu đồ 10 hoàn thành")

## 7. BÁO CÁO TÓMLƯỢC CUỐI CÙNG

In [ ]:
# Tạo báo cáo tổng hợp
print("="*70)
print("BÁO CÁO PHÂN TÍCH DỮ LIỆU LOTTO 5/35 - DASHBOARD V4")
print("="*70)

print(f"\n📊 THÔNG TIN DỮ LIỆU:")
print(f"  • Tổng số kỳ quay: {len(df)}")
print(f"  • Khoảng thời gian: {df['Ngày'].min().strftime('%d/%m/%Y')} đến {df['Ngày'].max().strftime('%d/%m/%Y')}")
print(f"  • Tổng số lần quay: {len(all_numbers)}")
print(f"  • Dãy số: 1-35")

print(f"\n📈 THỐNG KÊ TẦN SUẤT:")
print(f"  • Tần suất trung bình mỗi số: {len(all_numbers) / 35:.2f} lần")
print(f"  • Tần suất cao nhất: {frequency_df['Tần suất'].max()} (Số {frequency_df.iloc[0]['Số']})")
print(f"  • Tần suất thấp nhất: {frequency_df['Tần suất'].min()} (Số {frequency_df.iloc[-1]['Số']})")
print(f"  • Độ lệch chuẩn: {frequency_df['Tần suất'].std():.2f}")

print(f"\n🔢 THỐNG KÊ TỔNG SỐ:")
print(f"  • Tổng min: {sa_stats['sum_min']:.0f}")
print(f"  • Tổng max: {sa_stats['sum_max']:.0f}")
print(f"  • Tổng trung bình: {sa_stats['sum_mean']:.2f}")
print(f"  • Dự báo khoảng tổng (68% xác suất): {sa_predict['likely_range'][0]:.0f} - {sa_predict['likely_range'][1]:.0f}")
print(f"  • Dự báo khoảng tổng (95% xác suất): {sa_predict['confident_range'][0]:.0f} - {sa_predict['confident_range'][1]:.0f}")

print(f"\n🎯 TOP 10 SỐ DỰ BÁO:")
for rank, (num, score) in enumerate(top_predictions, 1):
    freq = number_frequency.get(num, 0)
    print(f"  {rank:2d}. Số {num:2d}: Điểm {score:6.2f}/100 | Tần suất: {freq} | Xác suất: {freq/len(all_numbers)*100:5.1f}%")

print(f"\n❄️  TOP 10 SỐ QUÁHẠN (Chưa xuất hiện lâu):")
for idx, (num, draws_ago) in enumerate(overdue_nums[:10], 1):
    print(f"  {idx:2d}. Số {num:2d}: {draws_ago} kỳ quay chưa xuất hiện")

print(f"\n💑 TOP 5 CẶP SỐ XUẤT HIỆN CÙNG NHAU:")
for idx, ((n1, n2), freq) in enumerate(sorted(pair_freq.items(), key=lambda x: x[1], reverse=True)[:5], 1):
    print(f"  {idx}. ({n1}, {n2}): {freq} lần")

print(f"\n📊 THỐNG KÊ KHOẢNG CÁCH:")
print(f"  • Khoảng cách phổ biến nhất: {ga_stats['most_common_gap']}")
print(f"  • Khoảng cách trung bình: {ga_stats['avg_gap']:.2f}")

print(f"\n⚡ GHI CHÚ QUAN TRỌNG:")
print(f"  • Phân tích này dựa trên thống kê xác suất")
print(f"  • Không đảm bảo kết quả 100% chính xác")
print(f"  • Nên kết hợp với kinh nghiệm và phân tích khác")
print(f"  • Tầm nhìn: Dành cho mục đích giáo dục và phân tích")

print("\n" + "="*70)

## 8. XUẤT DỮ LIỆU

In [ ]:
# Xuất kết quả ra CSV
import os

output_dir = 'analysis_results'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 1. Xuất tần suất tất cả số
frequency_export = pd.DataFrame([
    {'Số': num, 'Tần Suất': freq, 'Phần Trăm': f"{(freq / len(all_numbers) * 100):.2f}%", 
     'Điểm Dự Báo': all_scores[num]}
    for num, freq in sorted(number_frequency.items())
]).sort_values('Tần Suất', ascending=False)
frequency_export.to_csv(f'{output_dir}/frequency_analysis.csv', index=False, encoding='utf-8')

# 2. Xuất top predictions
top_pred_export = pd.DataFrame([
    {'Rank': rank, 'Số': num, 'Điểm Dự Báo': score, 'Tần Suất': number_frequency.get(num, 0)}
    for rank, (num, score) in enumerate(top_predictions, 1)
])
top_pred_export.to_csv(f'{output_dir}/top_predictions.csv', index=False, encoding='utf-8')

# 3. Xuất cặp số
pair_export = pd.DataFrame([
    {'Số 1': p[0], 'Số 2': p[1], 'Tần Suất': freq}
    for (p, freq) in sorted(pair_freq.items(), key=lambda x: x[1], reverse=True)[:50]
])
pair_export.to_csv(f'{output_dir}/pair_analysis.csv', index=False, encoding='utf-8')

# 4. Xuất số quá hạn
overdue_export = pd.DataFrame([
    {'Số': num, 'Kỳ Quay Chưa Xuất Hiện': draws_ago}
    for num, draws_ago in overdue_nums
])
overdue_export.to_csv(f'{output_dir}/overdue_numbers.csv', index=False, encoding='utf-8')

print(f"✓ Các file kết quả đã được xuất:")
print(f"  • frequency_analysis.csv")
print(f"  • top_predictions.csv")
print(f"  • pair_analysis.csv")
print(f"  • overdue_numbers.csv")

## KẾT LUẬN

Dashboard này sử dụng **7 thuật toán phân tích** để dự báo:

1. **Frequency Analysis** - Phân tích tần suất xuất hiện
2. **Overdue Analysis** - Phân tích số chưa xuất hiện lâu
3. **Pair Analysis** - Phân tích cặp số xuất hiện cùng nhau
4. **Statistical Prediction** - Dự báo dựa trên thống kê
5. **Positional Analysis** - Phân tích mẫu vị trí
6. **Sum & Average Analysis** - Phân tích tổng và trung bình
7. **Gap Analysis** - Phân tích khoảng cách giữa các số

**Các biểu đồ được hiển thị:**
- Tần suất của tất cả các số
- Top 15 số nóng & lạnh
- Điểm dự báo tổng hợp
- Tổng số qua các kỳ quay
- Phân bố tần suất
- Heatmap tần suất
- Top 20 cặp số
- Phân bố khoảng cách
- So sánh tần suất vs dự báo
- Pie chart phân loại số

**Lưu ý:** Phân tích này dành cho mục đích giáo dục và thống kê, không đảm bảo 100% chính xác.